In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
df = pd.read_csv("restaurant.csv")

In [ ]:
df.head()

,Order_ID,Customer_ID,Item_ID,Item_Name,Quantity,Order_Date
0,R0001,C001,F001,Burger,2,2026-10-01
1,R0001,C001,F002,Fries,1,2026-10-01
2,R0001,C001,F003,Coke,2,2026-10-01
3,R0001,C001,F009,Brownie,1,2026-10-01
4,R0001,C001,F011,Coffee,2,2026-10-01


In [ ]:
df.shape

(500, 6)

In [ ]:
df.isnull().sum()

,0
Order_ID,0
Customer_ID,0
Item_ID,0
Item_Name,0
Quantity,0
Order_Date,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
transactions = df.groupby('Order_ID')['Item_Name'].apply(list).tolist()

print(transactions[:5])

[['Burger', 'Fries', 'Coke', 'Brownie', 'Coffee'], ['Burger', 'Fries', 'Coke', 'Pizza', 'Salad'], ['Burger', 'Fries', 'Coke', 'Ice Cream', 'Brownie'], ['Burger', 'Fries', 'Coke', 'Pizza', 'Pasta'], ['Burger', 'Fries', 'Coke', 'Ice Cream', 'Brownie']]


In [ ]:
te = TransactionEncoder()

encoded = te.fit_transform(transactions)

basket = pd.DataFrame(encoded, columns=te.columns_)

basket.head()

,Brownie,Burger,Coffee,Coke,Fries,Garlic Bread,Ice Cream,Lemonade,Milkshake,Nuggets,Pasta,Pizza,Salad,Sandwich,Wrap
0,True,True,True,True,True,False,False,False,False,False,False,False,False,False,False
1,False,True,False,True,True,False,False,False,False,False,False,True,True,False,False
2,True,True,False,True,True,False,True,False,False,False,False,False,False,False,False
3,False,True,False,True,True,False,False,False,False,False,True,True,False,False,False
4,True,True,False,True,True,False,True,False,False,False,False,False,False,False,False


In [ ]:
frequent_items = apriori(
    basket,
    min_support=0.3,
    use_colnames=True
)
print(frequent_items)

    support               itemsets
0      0.72               (Burger)
1      0.32               (Coffee)
2      0.67                 (Coke)
3      0.75                (Fries)
4      0.34         (Garlic Bread)
5      0.41                (Pizza)
6      0.35             (Sandwich)
7      0.56         (Burger, Coke)
8      0.67        (Burger, Fries)
9      0.57          (Fries, Coke)
10     0.55  (Burger, Fries, Coke)


In [ ]:
rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.5
)

rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]]

,antecedents,consequents,support,confidence,lift
0,(Burger),(Coke),0.56,0.777778,1.160862
1,(Coke),(Burger),0.56,0.835821,1.160862
2,(Burger),(Fries),0.67,0.930556,1.240741
3,(Fries),(Burger),0.67,0.893333,1.240741
4,(Fries),(Coke),0.57,0.760000,1.134328
5,(Coke),(Fries),0.57,0.850746,1.134328
6,"(Burger, Fries)",(Coke),0.55,0.820896,1.225217
7,"(Burger, Coke)",(Fries),0.55,0.982143,1.309524
8,"(Fries, Coke)",(Burger),0.55,0.964912,1.340156
9,(Burger),"(Fries, Coke)",0.55,0.763889,1.340156


In [ ]:
rules = rules.sort_values(by="confidence", ascending=False)

rules[["antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"]]

,antecedents,consequents,support,confidence,lift
7,"(Burger, Coke)",(Fries),0.55,0.982143,1.309524
8,"(Fries, Coke)",(Burger),0.55,0.964912,1.340156
2,(Burger),(Fries),0.67,0.930556,1.240741
3,(Fries),(Burger),0.67,0.893333,1.240741
5,(Coke),(Fries),0.57,0.850746,1.134328
1,(Coke),(Burger),0.56,0.835821,1.160862
11,(Coke),"(Burger, Fries)",0.55,0.820896,1.225217
6,"(Burger, Fries)",(Coke),0.55,0.820896,1.225217
0,(Burger),(Coke),0.56,0.777778,1.160862
9,(Burger),"(Fries, Coke)",0.55,0.763889,1.340156
